# 01 — Data Audit
**Project:** AI-Powered E-Commerce Customer Churn Analytics & Retention Decision Support System  
**Author:** Sumit Raj  
**Dataset:** UCI Online Retail (`data/raw/Online Retail.xlsx`)  

**Purpose:** Profile the raw workbook, validate the schema, and document data-quality issues before any cleaning.  
All findings are descriptive — no rows are removed here.


In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve().parent))

import pandas as pd
from src.config import RAW_DATA_PATH, RAW_SHEET_NAME, EXPECTED_COLUMNS
from src.data.load import load_raw
from src.data.clean import audit_raw

## 1. Dataset Source


In [2]:
print(f'Raw data path : {RAW_DATA_PATH}')
print(f'Worksheet     : {RAW_SHEET_NAME}')
print(f'File exists   : {RAW_DATA_PATH.exists()}')

Raw data path : D:\IBM_SkillsBuild_Data_Analytics_AI_Internship_2026\ecommerce-churn-analytics\data\raw\Online Retail.xlsx
Worksheet     : Online Retail
File exists   : True


## 2. Loading


In [3]:
df_raw = load_raw()
print(f'Raw shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
df_raw.head(3)

Raw shape: 541,909 rows × 8 columns


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom


## 3. Schema Validation


In [4]:
print('Expected columns:', EXPECTED_COLUMNS)
print('Actual columns  :', list(df_raw.columns))
missing_cols = [c for c in EXPECTED_COLUMNS if c not in df_raw.columns]
print('Missing columns :', missing_cols if missing_cols else 'None — schema OK')

Expected columns: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']
Actual columns  : ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']
Missing columns : None — schema OK


## 4. Dimensions


In [5]:
print(f'Rows    : {df_raw.shape[0]:,}')
print(f'Columns : {df_raw.shape[1]}')

Rows    : 541,909
Columns : 8


## 5. Data Types


In [6]:
print(df_raw.dtypes)

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object


## 6. Date Range


In [7]:
print(f'InvoiceDate min : {df_raw["InvoiceDate"].min()}')
print(f'InvoiceDate max : {df_raw["InvoiceDate"].max()}')
print(f'Date parse failures: {df_raw["InvoiceDate"].isna().sum():,}')

InvoiceDate min : 2010-12-01 08:26:00
InvoiceDate max : 2011-12-09 12:50:00
Date parse failures: 0


## 7. Missing Values


In [8]:
missing = df_raw.isna().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
print(missing_df[missing_df['missing_count'] > 0].to_string())

             missing_count  missing_pct
Description           1454         0.27
CustomerID          135080        24.93


## 8. Exact Duplicates


In [9]:
n_dups = df_raw.duplicated().sum()
print(f'Exact duplicate rows : {n_dups:,}  ({n_dups/len(df_raw)*100:.3f}% of raw)')

Exact duplicate rows : 5,268  (0.972% of raw)


## 9. Cancellation Analysis


In [10]:
is_cancel = df_raw['InvoiceNo'].astype(str).str.startswith('C')
n_cancel = is_cancel.sum()
print(f'Cancellation rows (InvoiceNo starts with C) : {n_cancel:,}  ({n_cancel/len(df_raw)*100:.2f}%)')
print()
print('Top countries by cancellation row count:')
print(df_raw[is_cancel]['Country'].value_counts().head(5).to_string())

Cancellation rows (InvoiceNo starts with C) : 9,288  (1.71%)

Top countries by cancellation row count:
Country
United Kingdom    7856
Germany            453
EIRE               302
France             149
USA                112


## 10. Quantity Analysis


In [11]:
print(f'Quantity < 0  : {(df_raw["Quantity"] < 0).sum():,}')
print(f'Quantity == 0 : {(df_raw["Quantity"] == 0).sum():,}')
print(f'Quantity > 0  : {(df_raw["Quantity"] > 0).sum():,}')
print()
print('Quantity describe:')
print(df_raw['Quantity'].describe())

Quantity < 0  : 10,624
Quantity == 0 : 0
Quantity > 0  : 531,285

Quantity describe:
count    541909.000000
mean          9.552250
std         218.081158
min      -80995.000000
25%           1.000000
50%           3.000000
75%          10.000000
max       80995.000000
Name: Quantity, dtype: float64


## 11. UnitPrice Analysis


In [12]:
print(f'UnitPrice < 0  : {(df_raw["UnitPrice"] < 0).sum():,}')
print(f'UnitPrice == 0 : {(df_raw["UnitPrice"] == 0).sum():,}')
print(f'UnitPrice > 0  : {(df_raw["UnitPrice"] > 0).sum():,}')
print()
print('UnitPrice describe:')
print(df_raw['UnitPrice'].describe())

UnitPrice < 0  : 2
UnitPrice == 0 : 2,515
UnitPrice > 0  : 539,392

UnitPrice describe:


count    541909.000000
mean          4.611114
std          96.759853
min      -11062.060000
25%           1.250000
50%           2.080000
75%           4.130000
max       38970.000000
Name: UnitPrice, dtype: float64


## 12. Unique Counts


In [13]:
print(f'Unique InvoiceNo   : {df_raw["InvoiceNo"].nunique():,}')
print(f'Unique StockCode   : {df_raw["StockCode"].nunique():,}')
print(f'Unique CustomerID  : {df_raw["CustomerID"].nunique():,}  (excludes NaN)')
print(f'Unique Country     : {df_raw["Country"].nunique():,}')
print()
print('Top 5 countries by row count:')
print(df_raw['Country'].value_counts().head(5).to_string())

Unique InvoiceNo   : 25,900
Unique StockCode   : 4,070
Unique CustomerID  : 4,372  (excludes NaN)
Unique Country     : 38

Top 5 countries by row count:
Country
United Kingdom    495478
Germany             9495
France              8557
EIRE                8196
Spain               2533


## 13. Full Audit Summary (via audit_raw)


In [14]:
report = audit_raw(df_raw)
print('=== Data Quality Audit Summary ===')
for k, v in report.items():
    print(f'  {k}: {v}')

=== Data Quality Audit Summary ===
  raw_rows: 541909
  raw_columns: 8
  missing_customer_id: 135080
  missing_description: 1454
  exact_duplicates: 5268
  cancellation_rows: 9288
  negative_quantity_rows: 10624
  zero_quantity_rows: 0
  non_positive_quantity_rows: 10624
  negative_unit_price_rows: 2
  zero_unit_price_rows: 2515
  non_positive_unit_price_rows: 2517
  date_parse_failures: 0
  date_min: 2010-12-01
  date_max: 2011-12-09
  unique_invoices: 25900
  unique_products: 4070
  unique_customers: 4372
  country_count: 38


## 14. Data-Quality Conclusions

| Issue | Count | % of Raw |
|---|---|---|
| Raw transaction rows | 541,909 | 100% |
| Missing CustomerID | 135,080 | 24.9% |
| Missing Description | 1,454 | 0.27% |
| Exact duplicate rows | 5,268 | 0.97% |
| Cancellation rows (InvoiceNo starts 'C') | 9,288 | 1.71% |
| Negative Quantity rows | 10,624 | 1.96% |
| Zero Quantity rows | 0 | 0% |
| Negative UnitPrice rows | 2 | <0.01% |
| Zero UnitPrice rows | 2,515 | 0.46% |

**Key observations:**
- ~25% of rows have no CustomerID — these are guest/anonymous transactions and cannot be attributed to a specific customer. They must be excluded from the customer-level analytical dataset but are retained in the raw audit.
- Missing Description does not invalidate a transaction; will be represented as 'Unknown' for product analysis.
- Cancellation rows (InvoiceNo prefix 'C') account for 1.71% of the dataset — these must be excluded from qualifying purchases.
- Negative Quantity rows overlap significantly with cancellation rows (same 'C' prefix InvoiceNos).
- Date range spans 2010-12-01 through 2011-12-09 — covers the full observation and future windows needed for churn labelling.
- No date parse failures — InvoiceDate parsed cleanly.
